In [ ]:
import pandas as pd
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import seaborn as sns
import matplotlib.pyplot as plt

# 1. LOAD DATA
df = pd.read_csv('data/processed/training_data_final.csv')
features = ['latitude', 'longitude', 'month_sin', 'month_cos', 'temp', 'humidity', 'wind', 'vpd']
X = df[features]; y = df['fire_detected']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. DEFINE MODELS
cat = CatBoostClassifier(iterations=1000, depth=8, learning_rate=0.05, verbose=0)
xg = xgb.XGBClassifier(n_estimators=1000, max_depth=8, learning_rate=0.05)

ensemble = VotingClassifier(estimators=[('cat', cat), ('xg', xg)], voting='soft')

# 3. TRAIN
print("🤖 Fitting Ensemble...")
ensemble.fit(X_train, y_train)

# 4. EVALUATE
print("\n🏆 Final Metrics:")
print(classification_report(y_test, ensemble.predict(X_test)))

# Save
os.makedirs('models', exist_ok=True)
joblib.dump(ensemble, 'models/wildfire_ensemble.pkl')
print("✅ Model Saved: models/wildfire_ensemble.pkl")